# Experiment: Classification model comparison for Challenge A

Objective:
- Follow the brief more closely by predicting late refill risk.
- Compare Logistic Regression, Random Forest, and XGBoost on `target_is_late_refill_7d`.
- Use `PR-AUC` as the main ranking metric and add a quick calibration check.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)
        


## 1. Load the preprocessing output

We reuse the output from `preprocesing.ipynb` and add a few light classification-oriented helper features.


In [ ]:
ROOT = Path.cwd()
DATA_PATH = ROOT / "DATA-AI-Hackathon-Track-1" / "data" / "processed" / "challenge_a_classification_dataset.csv"
OUTPUT_DIR = ROOT / "DATA-AI-Hackathon-Track-1" / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["service_date"] = pd.to_datetime(df["service_date"])
df["run_out_date"] = pd.to_datetime(df["run_out_date"])

df["prev_fill_was_early"] = (df["gap_from_previous_fill_days"] < 0).astype(int)
df["prev_fill_was_late_7d"] = (df["gap_from_previous_fill_days"] > 7).astype(int)
df["abs_gap_from_previous_fill_days"] = df["gap_from_previous_fill_days"].abs()

print(f"Rows: {len(df):,}")
print(f"Date range: {df['service_date'].min().date()} -> {df['service_date'].max().date()}")
df.head()


## 2. Define target, features, and temporal split

We follow the brief with a fully time-based split:
- train: 2008-01-01 through 2009-12-31
- validation: 2010-01-01 through 2010-06-30
- test: 2010-07-01 through 2010-12-31

This lets us select both the model and the decision threshold on an earlier future window, then keep the last 6 months as a clean final test.
        


In [ ]:
TARGET_COL = "target_is_late_refill_7d"
FUTURE_OR_TARGET_COLS = ["target_days_until_next_fill", "target_refill_gap_days", TARGET_COL]
ID_AND_DATE_COLS = [
    "DESYNPUF_ID",
    "PDE_ID",
    "drug_group_key",
    "ndc9",
    "PROD_SRVC_ID",
    "ndc11",
    "service_date",
    "run_out_date",
]
DROP_COLS = set(ID_AND_DATE_COLS + FUTURE_OR_TARGET_COLS)

In [ ]:
# Calculate positive rate by quarter for the entire date range
tmp = df.copy()
tmp["year"] = tmp["service_date"].dt.year
tmp["quarter"] = tmp["service_date"].dt.quarter

quarterly_positive_rate = (
    tmp.groupby(["year", "quarter"], as_index=False)
    .agg(
        positive_rate=(TARGET_COL, "mean"),
        total_records=(TARGET_COL, "size"),
        positive_count=(TARGET_COL, "sum"),
    )
)

quarterly_positive_rate["year_quarter"] = (
    quarterly_positive_rate["year"].astype(str)
    + "-Q"
    + quarterly_positive_rate["quarter"].astype(str)
)

quarterly_positive_rate


In [ ]:

feature_cols = [
    col for col in df.columns
    if col not in DROP_COLS and pd.api.types.is_numeric_dtype(df[col])
]

TRAIN_END = pd.Timestamp("2010-01-01")
VAL_END = pd.Timestamp("2010-07-01")

train_mask = df["service_date"] < TRAIN_END
val_mask = (df["service_date"] >= TRAIN_END) & (df["service_date"] < VAL_END)
test_mask = df["service_date"] >= VAL_END

X_train = df.loc[train_mask, feature_cols].copy()
X_val = df.loc[val_mask, feature_cols].copy()
X_test = df.loc[test_mask, feature_cols].copy()
y_train = df.loc[train_mask, TARGET_COL].astype(int).copy()
y_val = df.loc[val_mask, TARGET_COL].astype(int).copy()
y_test = df.loc[test_mask, TARGET_COL].astype(int).copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_val), len(X_test)],
        "date_min": [
            df.loc[train_mask, "service_date"].min(),
            df.loc[val_mask, "service_date"].min(),
            df.loc[test_mask, "service_date"].min(),
        ],
        "date_max": [
            df.loc[train_mask, "service_date"].max(),
            df.loc[val_mask, "service_date"].max(),
            df.loc[test_mask, "service_date"].max(),
        ],
        "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    }
)

print(f"Number of numeric features: {len(feature_cols)}")
split_summary


## 3. Build the three classifiers


In [ ]:
RANDOM_STATE = 42

models = {
    "Logistic Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=5,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    n_estimators=400,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

list(models.keys())


## 4. Compare models on the validation window

Primary selection metric:
- `PR-AUC`

Supporting checks:
- `ROC-AUC`
- `Brier score`

We use only the validation period to choose the model.
        


In [ ]:
fitted_validation_models = {}
validation_results = []

validation_prediction_frame = df.loc[val_mask, [
    "service_date",
    "DESYNPUF_ID",
    "drug_group_key",
    TARGET_COL,
]].copy()
validation_prediction_frame = validation_prediction_frame.rename(
    columns={TARGET_COL: "actual_late_refill_7d"}
)

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_val)[:, 1]
    fitted_validation_models[model_name] = pipeline
    validation_prediction_frame[f"proba_{model_name.lower().replace(' ', '_')}"] = proba

    validation_results.append(
        {
            "model": model_name,
            "pr_auc": average_precision_score(y_val, proba),
            "roc_auc": roc_auc_score(y_val, proba),
            "brier": brier_score_loss(y_val, proba),
            "mean_predicted_risk": float(proba.mean()),
            "observed_positive_rate": float(y_val.mean()),
        }
    )

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)
validation_results_df
        


## 5. Select a decision threshold on validation

Once the top validation model is selected, we sweep a range of thresholds and pick the one with the best validation `F1`, using recall and precision as tie-breakers.
        


In [ ]:
best_model_name = validation_results_df.iloc[0]["model"]
best_val_proba_col = f"proba_{best_model_name.lower().replace(' ', '_')}"
best_val_proba = validation_prediction_frame[best_val_proba_col]

threshold_grid = np.round(np.arange(0.01, 0.51, 0.01), 2)
threshold_rows = []

for threshold in threshold_grid:
    pred = (best_val_proba >= threshold).astype(int)
    threshold_rows.append(
        {
            "model": best_model_name,
            "threshold": threshold,
            "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
            "predicted_positive_rate": float(pred.mean()),
            "predicted_positives": int(pred.sum()),
        }
    )

threshold_results_df = (
    pd.DataFrame(threshold_rows)
    .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)
selected_threshold = float(threshold_results_df.iloc[0]["threshold"])

print(f"Best validation model: {best_model_name}")
print(f"Selected threshold: {selected_threshold:.2f}")
threshold_results_df.head(15)
        


## 6. Refit on train plus validation and evaluate on the final test

We freeze the model choice and threshold from validation, refit on `train + validation`, and evaluate once on the final 6-month test window.
        


In [ ]:
dev_mask = train_mask | val_mask
X_dev = df.loc[dev_mask, feature_cols].copy()
y_dev = df.loc[dev_mask, TARGET_COL].astype(int).copy()

fitted_test_models = {}
test_results = []

for model_name, pipeline in models.items():
    pipeline.fit(X_dev, y_dev)
    proba = pipeline.predict_proba(X_test)[:, 1]
    fitted_test_models[model_name] = pipeline
    test_results.append(
        {
            "model": model_name,
            "pr_auc": average_precision_score(y_test, proba),
            "roc_auc": roc_auc_score(y_test, proba),
            "brier": brier_score_loss(y_test, proba),
            "mean_predicted_risk": float(proba.mean()),
            "observed_positive_rate": float(y_test.mean()),
        }
    )

test_results_df = (
    pd.DataFrame(test_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)
test_results_df
        


In [ ]:
selected_test_pipeline = fitted_test_models[best_model_name]
test_proba = selected_test_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

best_test_prediction_frame = df.loc[test_mask, [
    "service_date",
    "run_out_date",
    "DESYNPUF_ID",
    "drug_group_key",
    "ndc9",
    "PROD_SRVC_ID",
    TARGET_COL,
]].copy()
best_test_prediction_frame = best_test_prediction_frame.rename(
    columns={TARGET_COL: "actual_late_refill_7d"}
)
best_test_prediction_frame["predicted_risk"] = test_proba
best_test_prediction_frame["predicted_late_refill_7d"] = test_pred
best_test_prediction_frame["selected_model"] = best_model_name
best_test_prediction_frame["selected_threshold"] = selected_threshold

best_model_test_metrics_df = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "threshold": selected_threshold,
            "pr_auc": average_precision_score(y_test, test_proba),
            "roc_auc": roc_auc_score(y_test, test_proba),
            "brier": brier_score_loss(y_test, test_proba),
            "precision": precision_score(y_test, test_pred, zero_division=0),
            "recall": recall_score(y_test, test_pred, zero_division=0),
            "f1": f1_score(y_test, test_pred, zero_division=0),
            "mean_predicted_risk": float(test_proba.mean()),
            "observed_positive_rate": float(y_test.mean()),
            "predicted_positive_rate": float(test_pred.mean()),
            "predicted_positives": int(test_pred.sum()),
        }
    ]
)

best_model_test_metrics_df
        


## 7. Quick calibration check

For the selected model, compare average predicted probability against observed event rate by probability decile on the final test window.
        


In [ ]:
calib = best_test_prediction_frame[["actual_late_refill_7d", "predicted_risk"]].copy()
calib["bin"] = pd.qcut(
    calib["predicted_risk"].rank(method="first"),
    q=10,
    labels=False,
    duplicates="drop",
)

best_model_calibration_df = (
    calib.groupby("bin")
    .agg(
        mean_predicted_risk=("predicted_risk", "mean"),
        observed_rate=("actual_late_refill_7d", "mean"),
        n=("actual_late_refill_7d", "size"),
    )
    .reset_index()
)
best_model_calibration_df["model"] = best_model_name
best_model_calibration_df
        


## 8. Demo slice: patient timeline and risk score

We pick one patient from the highest-risk predictions of the selected validation winner and show the timeline for up to two drug groups in the final test window.
        


In [ ]:
demo_patient_id = (
    best_test_prediction_frame.sort_values("predicted_risk", ascending=False)
    .iloc[0]["DESYNPUF_ID"]
)

patient_timeline = best_test_prediction_frame.loc[
    best_test_prediction_frame["DESYNPUF_ID"] == demo_patient_id,
    [
        "service_date",
        "run_out_date",
        "DESYNPUF_ID",
        "drug_group_key",
        "ndc9",
        "PROD_SRVC_ID",
        "actual_late_refill_7d",
        "predicted_risk",
        "predicted_late_refill_7d",
    ],
].copy()

top_two_drugs = patient_timeline["drug_group_key"].value_counts().head(2).index
patient_timeline = patient_timeline.loc[
    patient_timeline["drug_group_key"].isin(top_two_drugs)
].sort_values(["drug_group_key", "service_date"])

patient_timeline
        


## 9. Top drivers

For the selected model, show a quick global ranking of the most influential features after refitting on `train + validation`.
        


In [ ]:
best_pipeline = selected_test_pipeline
best_estimator = best_pipeline.named_steps["model"]

if best_model_name == "Logistic Regression":
    importance_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "signed_weight": best_estimator.coef_[0],
            "importance": np.abs(best_estimator.coef_[0]),
        }
    ).sort_values("importance", ascending=False)
else:
    importance_df = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": best_estimator.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

importance_df.head(20)
        


## 10. Save outputs

Save validation comparisons, threshold search, final test metrics, selected-model predictions, calibration summaries, and top-driver tables.
        


In [ ]:
validation_comparison_path = OUTPUT_DIR / "classification_validation_model_comparison.csv"
test_comparison_path = OUTPUT_DIR / "classification_test_model_comparison.csv"
threshold_path = OUTPUT_DIR / "classification_threshold_search.csv"
best_metrics_path = OUTPUT_DIR / "classification_best_model_test_metrics.csv"
predictions_path = OUTPUT_DIR / "classification_best_model_test_predictions.csv"
calibration_path = OUTPUT_DIR / "classification_best_model_calibration.csv"
importance_path = OUTPUT_DIR / "classification_best_model_top_drivers.csv"

validation_results_df.to_csv(validation_comparison_path, index=False)
test_results_df.to_csv(test_comparison_path, index=False)
threshold_results_df.to_csv(threshold_path, index=False)
best_model_test_metrics_df.to_csv(best_metrics_path, index=False)
best_test_prediction_frame.to_csv(predictions_path, index=False)
best_model_calibration_df.to_csv(calibration_path, index=False)
importance_df.to_csv(importance_path, index=False)

(
    validation_comparison_path,
    test_comparison_path,
    threshold_path,
    best_metrics_path,
    predictions_path,
    calibration_path,
    importance_path,
)
        
